# Experiment 1 — Loan Prediction Classification

**Course** : Machine Learning Laboratory — Semester 5  
**Roll No** : 3122247001061  
**Dataset** : Loan Prediction Dataset (Analytics Vidhya)  
**Objective**: Build and evaluate classification models to predict whether a loan application will be approved (Y) or rejected (N) based on applicant demographics and financial information.

---

**Pipeline**

```
Configuration  →  Load Dataset  →  EDA  →  Preprocessing
→  Feature Selection  →  Train-Test Split  →  Model Training
→  Evaluation  →  Save Outputs  →  Observations
```

In [ ]:
import sys
import os

# ---------------------------------------------------------------------------
# PATH SETUP
# ---------------------------------------------------------------------------
# Add ../src/ to sys.path so the notebook can import from the reusable
# framework (eda.py, preprocessing.py, feature_selection.py, models.py,
# evaluation.py) without installing them as packages.
# os.path.abspath resolves the relative path from the notebook's location.

SRC_PATH = os.path.abspath(os.path.join("..", "src"))
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# ---------------------------------------------------------------------------
# DATASET CONFIGURATION
# ---------------------------------------------------------------------------
# DATASET_PATH     : Path to the raw CSV file relative to this notebook.
# TARGET_COLUMN    : Name of the label column (the column the model will predict).
# EXCLUDE_COLUMNS  : Identifier columns to exclude from all transformations.
#                    These are kept in the DataFrame but excluded from EDA
#                    analysis, preprocessing, and model training.
#                    Drop these in the Feature Selection step.

DATASET_PATH    = os.path.join("..", "dataset", "loan_prediction", "train.csv")
TARGET_COLUMN   = "Loan_Status"
EXCLUDE_COLUMNS = ["Loan_ID"]

# ---------------------------------------------------------------------------
# OUTPUT PATHS
# ---------------------------------------------------------------------------
# FIGURES_PATH : Directory where EDA plots are saved as EPS files at 600 DPI.
#                Created automatically if it does not exist.

FIGURES_PATH = os.path.abspath(os.path.join("..", "figures"))

# ---------------------------------------------------------------------------
# TRAIN-TEST SPLIT CONFIGURATION
# ---------------------------------------------------------------------------
# TEST_SIZE    : Fraction of the dataset reserved for the test set.
#                0.20 means 80% training / 20% testing.
# RANDOM_STATE : Seed for reproducibility. Fix this value so that every run
#                produces the same train-test split.

TEST_SIZE    = 0.20
RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# PREPROCESSING STRATEGY CHOICES
# ---------------------------------------------------------------------------
# These are passed directly to classification_preprocessing() in the
# Preprocessing cell. Choices are made here (not scattered in code) so
# the rationale is documented in one place.
#
# MISSING_STRATEGY : "mean" -- suitable for approximately normal distributions.
#                    Categorical NaN columns always use mode regardless.
# ENCODING         : "label" -- assigns an integer to each unique category.
#                    Appropriate for binary features in this dataset.
# SCALING          : None -- tree-based models (Decision Tree, Naive Bayes)
#                    are scale-invariant; scaling is not required.

MISSING_STRATEGY = "mean"
ENCODING         = "label"
SCALING          = None

# ---------------------------------------------------------------------------
# FEATURE SELECTION CONFIGURATION
# ---------------------------------------------------------------------------
# FS_METHOD : Scoring method passed to classification_feature_selection().
#             "chi2"        -- Chi-Square test (non-negative values required).
#             "anova"       -- ANOVA F-test (any numerical features).
#             "mutual_info" -- Mutual Information (detects non-linear links).
#             chi2 is appropriate here: label-encoded features are non-negative.
# FS_K      : Number of top features to retain.

FS_METHOD = "chi2"
FS_K      = 8

# ---------------------------------------------------------------------------
# CONFIRMATION PRINT
# ---------------------------------------------------------------------------
print("=" * 60)
print("  EXPERIMENT CONFIGURATION")
print("=" * 60)
print(f"  SRC Path         : {SRC_PATH}")
print(f"  Dataset Path     : {DATASET_PATH}")
print(f"  Target Column    : {TARGET_COLUMN}")
print(f"  Exclude Columns  : {EXCLUDE_COLUMNS}")
print(f"  Figures Path     : {FIGURES_PATH}")
print(f"  Test Size        : {TEST_SIZE}")
print(f"  Random State     : {RANDOM_STATE}")
print(f"  Missing Strategy : {MISSING_STRATEGY}")
print(f"  Encoding         : {ENCODING}")
print(f"  Scaling          : {SCALING}")
print(f"  FS Method        : {FS_METHOD}")
print(f"  FS k             : {FS_K}")
print("=" * 60)
print("  Configuration complete. Proceed to Data Loading.")

In [ ]:
import pandas as pd

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
# pd.read_csv() is the only operation here.
# DATASET_PATH is defined in the Configuration cell (cell 2).
# The result is the RAW dataset -- no transformation is applied.
# EDA must always receive the raw data so it reflects the true
# state of the dataset before any cleaning decisions are made.

df = pd.read_csv(DATASET_PATH)

# ---------------------------------------------------------------------------
# VALIDATE
# ---------------------------------------------------------------------------
# Guard 1 -- The file must have loaded at least one row.
#            An empty DataFrame here means the CSV path is wrong
#            or the file is corrupted. Fail immediately with a
#            descriptive message rather than a cryptic error later.

assert not df.empty, (
    f"Dataset loaded from '{DATASET_PATH}' is empty. "
    f"Check that the file exists and is a valid CSV."
)

# Guard 2 -- The target column must exist.
#            classification_eda() and classification_preprocessing()
#            both require it. A missing target column here means the
#            wrong CSV was loaded or TARGET_COLUMN is mis-spelled.

assert TARGET_COLUMN in df.columns, (
    f"Target column '{TARGET_COLUMN}' not found in dataset.\n"
    f"Available columns: {df.columns.tolist()}"
)

# Guard 3 -- All declared identifier columns must exist.
#            EXCLUDE_COLUMNS are passed to preprocessing as protected
#            columns. A mis-spelled identifier causes a silent failure
#            where the column is treated as a feature instead.

missing_ids = [col for col in EXCLUDE_COLUMNS if col not in df.columns]
assert not missing_ids, (
    f"EXCLUDE_COLUMNS {missing_ids} not found in dataset.\n"
    f"Available columns: {df.columns.tolist()}"
)

# ---------------------------------------------------------------------------
# SUMMARY PRINT
# ---------------------------------------------------------------------------
print("=" * 60)
print("  DATASET LOADED")
print("=" * 60)
print(f"  File             : {DATASET_PATH}")
print(f"  Shape            : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Target Column    : '{TARGET_COLUMN}'")
print(f"  Target Classes   : {df[TARGET_COLUMN].unique().tolist()}")
print(f"  Identifier Cols  : {EXCLUDE_COLUMNS}")
print(f"  Memory Usage     : {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
print("=" * 60)
print("  All validation guards passed. df is ready for EDA.")
df.head()

In [ ]:
from eda import classification_eda

eda_results = classification_eda(
    df=df,
    target_column=TARGET_COLUMN,
    figures_path=FIGURES_PATH
)

print("EDA completed successfully. Observations stored in eda_results.")

In [ ]:
from preprocessing import classification_preprocessing

# ---------------------------------------------------------------------------
# PREPROCESS
# ---------------------------------------------------------------------------
# classification_preprocessing() is the ONLY function called from
# preprocessing.py. It internally orchestrates four steps in order:
#
#   Step 1 -- Handle Duplicates    (strategy: duplicate_strategy="drop")
#   Step 2 -- Handle Missing Values (strategy: MISSING_STRATEGY)
#   Step 3 -- Encode Categoricals   (method: ENCODING)
#   Step 4 -- Scale Numericals      (method: SCALING)
#
# All strategy choices are sourced from Cell 2 (Configuration).
# The original df is NEVER modified -- the module works on an internal copy.
# TARGET_COLUMN and EXCLUDE_COLUMNS are fully protected from all steps.

clean_df = classification_preprocessing(
    df=df,
    target_column=TARGET_COLUMN,
    exclude_columns=EXCLUDE_COLUMNS,
    missing_strategy=MISSING_STRATEGY,
    encoding=ENCODING,
    scaling=SCALING,
)

# ---------------------------------------------------------------------------
# SUMMARY PRINT
# ---------------------------------------------------------------------------
# classification_preprocessing() already prints a detailed step-by-step log.
# This block adds a concise post-call confirmation: shape change, NaN count,
# and the column list after encoding -- so the reader can verify at a glance
# that encoding and duplicate removal worked as expected.

print("=" * 60)
print("  PREPROCESSING SUMMARY")
print("=" * 60)
print(f"  Input shape      : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Output shape     : {clean_df.shape[0]} rows x {clean_df.shape[1]} columns")
print(f"  Rows removed     : {df.shape[0] - clean_df.shape[0]}  (duplicates + dropped NaN rows)")
print(f"  Remaining NaN    : {clean_df.isnull().sum().sum()}")
print(f"  Columns          : {clean_df.columns.tolist()}")
print("=" * 60)
print("  clean_df is ready for Feature Selection.")
clean_df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from feature_selection import classification_feature_selection

# ---------------------------------------------------------------------------
# PRE-CALL REQUIREMENT: Encode the target column to integer
# ---------------------------------------------------------------------------
# classification_preprocessing() protects TARGET_COLUMN from encoding by
# design (encoding only transforms feature columns). As a result, Loan_Status
# is still a string ("Y" / "N") in clean_df.
#
# classification_feature_selection() calls sklearn scoring functions
# (chi2, f_classif, mutual_info_classif) which require a NUMERIC target.
# The module raises a ValueError if the target is non-numeric.
# Its docstring (see: feature_selection.py, Notes section) explicitly
# instructs the notebook caller to encode the target here.
#
# This is NOT a preprocessing step -- it is fulfilling the documented
# API contract of classification_feature_selection().

le_target                       = LabelEncoder()
clean_df[TARGET_COLUMN]         = le_target.fit_transform(clean_df[TARGET_COLUMN])
target_encoding_map             = dict(zip(
    le_target.classes_,
    le_target.transform(le_target.classes_).tolist()
))

print(f"  Target encoding  : {target_encoding_map}")
print(f"  (e.g., N=0 means loan rejected; Y=1 means loan approved)")
print()

# ---------------------------------------------------------------------------
# FEATURE SELECTION
# ---------------------------------------------------------------------------
# classification_feature_selection() is the ONLY function called from
# feature_selection.py. It internally:
#   Step 0 -- Validates all inputs
#   Step 1 -- Extracts the feature matrix and target vector
#   Step 2 -- Applies a variance pre-filter (removes constant features)
#   Step 3 -- Auto-caps k if it exceeds available features
#   Step 4 -- Scores features using FS_METHOD (chi2 here)
#   Step 5 -- Builds the feature ranking DataFrame
#   Step 6 -- Prints the full ranking table
#   Step 7 -- Builds selected_df (selected features + target + identifiers)
#
# FS_METHOD and FS_K are sourced from Cell 2 (Configuration).
# EXCLUDE_COLUMNS (Loan_ID) is passed so identifiers are excluded from
# scoring but kept in selected_df for reference.

selected_df, selected_features, feature_ranking_df = classification_feature_selection(
    df=clean_df,
    target_column=TARGET_COLUMN,
    method=FS_METHOD,
    k=FS_K,
    exclude_columns=EXCLUDE_COLUMNS,
)

# ---------------------------------------------------------------------------
# EXTRACT X AND y
# ---------------------------------------------------------------------------
# The module's docstring explicitly instructs the caller to build X and y
# from selected_df after the call. This is the documented usage pattern:
#
#   X = selected_df[selected_features]
#   y = selected_df[target_column]
#
# X contains ONLY the top-k selected feature columns (no target, no Loan_ID).
# y is the target vector (0/1 integers) aligned row-for-row with X.
# Both are passed to train_test_split() in Cell 7.

X = selected_df[selected_features]
y = selected_df[TARGET_COLUMN]

# ---------------------------------------------------------------------------
# SUMMARY PRINT
# ---------------------------------------------------------------------------
print("=" * 60)
print("  FEATURE SELECTION SUMMARY")
print("=" * 60)
print(f"  Method           : {FS_METHOD}")
print(f"  Features before  : {clean_df.shape[1] - 1 - len(EXCLUDE_COLUMNS)}  (excluding target and identifiers)")
print(f"  Features selected: {len(selected_features)}")
print(f"  Selected features: {selected_features}")
print(f"  X shape          : {X.shape}")
print(f"  y shape          : {y.shape}  |  classes: {sorted(y.unique().tolist())}")
print("=" * 60)
print("  X and y are ready for train_test_split() in Cell 7.")
print()

# ---------------------------------------------------------------------------
# DISPLAY
# ---------------------------------------------------------------------------
print("Feature Ranking:")
display(feature_ranking_df)

print("\nX.head():")
display(X.head())

print("\ny.head():")
display(y.head())

In [ ]:
# STAGE 7 — Train-Test Split
# Implementation: next cell to be written.

In [ ]:
# STAGE 8 — Model Training
# Implementation: next cell to be written.

In [ ]:
# STAGE 9 — Performance Evaluation
# Implementation: next cell to be written.

In [ ]:
# STAGE 10 — Save Outputs
# Implementation: next cell to be written.

## Student Observations

*Write your manual observations here after all pipeline stages have been executed.*

- **EDA Observations** :
- **Preprocessing Decisions** :
- **Feature Selection Rationale** :
- **Model Comparison** :
- **Final Conclusion** :